In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
#anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
#deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
#grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

#anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [5]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [6]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the aspiring LLM engineer bring a ladder to the lab?

Because they heard they needed to work on *deep* learning!

## Training vs Inference time scaling

In [7]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [8]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/3

In [9]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

2/3

## Testing out the best models on the planet

In [11]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [12]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

We have two volumes with the following dimensions:
- Each volume: pages thickness = 2 cm (per volume)
- Each cover thickness = 2 mm = 0.2 cm
- They are on a shelf side by side, in the order: first volume then second volume.

A worm gnaws perpendicular to the pages from the first page of the first volume to the last page of the second volume. That means it starts at the very front of the first volume’s pages (the first page) and ends at the very back of the second volume’s pages (the last page).

To determine the total gnawed distance, sum the thicknesses along the path through the intervening material:

Path from:
- the first page of the first volume to the back cover of the first volume
  - This includes the front cover of the first volume + its pages + its back cover, but since the worm starts on the first page (which is immediately after the front cover), it must go through the remainder of the first volume up to its back page edge.
  However, a simpler way is to compute the total thickness from the starting surface to the ending surface along the shelf line.

Set up coordinates along the shelf from the left:
- Front cover of volume 1: 0 to 0.2 cm
- Pages of volume 1: 0.2 cm to 0.2 + 2.0 = 2.2 cm
- Back cover of volume 1: 2.2 to 2.4 cm
- Gap between volumes is zero (they are touching on the shelf)
- Front cover of volume 2: 2.4 to 2.6 cm
- Pages of volume 2: 2.6 to 4.6 cm
- Back cover of volume 2: 4.6 to 4.8 cm

The worm starts at the first page of the first volume, which lies at the inner boundary of the front side of the pages: that is at the position 0.2 cm (the start of the pages). It ends at the last page of the second volume, which lies at the boundary just before the back cover of volume 2: that is at position 4.6 cm (the start of the back cover).

Distance gnawed = ending position − starting position = 4.6 − 0.2 = 4.4 cm.

Therefore, the worm gnawed through 4.4 cm of material.

In [14]:
response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

This is a classic riddle that plays on our assumptions about how books are arranged on a shelf. Let's break it down by visualizing the books' positions.

1.  **The Setup:** The books are side by side in the usual order: Volume 1 on the left, Volume 2 on the right.

    `[Volume 1] [Volume 2]`

2.  **The Anatomy of a Book on a Shelf:**
    *   For **Volume 1**, the front cover is on the right (facing Volume 2), and the back cover is on the left. The first page is just inside the front cover.
    *   For **Volume 2**, the front cover is on the left (facing Volume 1), and the back cover is on the right. The last page is just inside the back cover.

3.  **Visualizing the Worm's Path:**
    *   The worm starts at the **"first page of the first volume."** This page is on the far right side of the pages of Volume 1, right next to Volume 2.
    *   The worm ends at the **"last page of the second volume."** This page is on the far right side of the pages of Volume 2, just before the back cover.

    Let's map out what the worm gnaws through in a straight line from start to finish:

    `[BackCov1 | PagesV1 | FrontCov1] [FrontCov2 | PagesV2 | BackCov2]`
                       `^`                             `^`
                     `START`                           `END`

    The worm doesn't travel through the pages of Volume 1. It starts on the first page and moves right. The path goes through:
    1.  The front cover of Volume 1 (**2 mm**)
    2.  The front cover of Volume 2 (**2 mm**)
    3.  All the pages of Volume 2 (**2 cm**)

4.  **The Calculation:**
    *   We need to add the thicknesses of everything the worm gnawed through.
    *   First, let's make the units consistent. 2 cm = 20 mm.
    *   Total distance = (Cover 1) + (Cover 2) + (Pages of V2)
    *   Total distance = 2 mm + 2 mm + 20 mm = 24 mm

Converting back to centimeters:

**The worm gnawed through 2.4 cm.**

## A spicy challenge to test the competitive spirit

In [16]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [17]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))

**I would choose “Share.”**  

In this version of the Prisoner’s Dilemma the payoff matrix looks like this:

|                | Partner Shares | Partner Steals |
|----------------|----------------|----------------|
| **You Share**  | You $1,000 – Partner $1,000 | You $0 – Partner $2,000 |
| **You Steal**  | You $2,000 – Partner $0   | You $0 – Partner $0   |

If we think only about a single isolated round with no future interaction, the **dominant strategy** (the move that gives the highest payoff regardless of what the other does) is to **Steal**, because:

- If the partner shares, stealing gives you $2,000 vs. $1,000 for sharing.
- If the partner steals, stealing still gives you $0 (the same as sharing).

However, the **jointly optimal** outcome—where both players get the most total money—is for both to **Share**, yielding $2,000 total ($1,000 each) rather than $0 total when both steal.

Choosing “Share” signals trust and aims for the best collective result. In many real‑world game‑show settings, contestants often try to coordinate (e.g., by pre‑show agreements or subtle cues) to achieve that mutually beneficial outcome. So, assuming I’m hoping for the highest guaranteed payout for both of us, I’d pick **Share**.

In [21]:
response = deepseek.chat.completions.create(model="meta-llama/llama-3.3-70b-instruct:free", messages=dilemma)
display(Markdown(response.choices[0].message.content))

A classic Prisoner's Dilemma!

I choose... Share.

My reasoning is that if I assume my partner is a rational and reasonable person, we'll both be better off if we both choose to Share. We'll each get $1,000, which is a guaranteed payout. If I choose to Steal and my partner also chooses to Steal, we'll both end up with nothing, which is a worse outcome. And if I choose to Steal and my partner chooses to Share, while I might get $2,000, I'd rather not take the risk of getting nothing and instead opt for the guaranteed $1,000.

Plus, choosing to Share shows that I'm willing to cooperate and trust my partner, which could lead to a more positive outcome for both of us. So, I'll take the high road and choose to Share!

In [22]:
response = grok.chat.completions.create(model="x-ai/grok-4.1-fast:free", messages=dilemma)
display(Markdown(response.choices[0].message.content))

Steal

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

In [ ]:
!ollama pull llama3.2

In [ ]:
# Only do this if you have a large machine - at least 16GB RAM

!ollama pull gpt-oss:20b

In [24]:
response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

1/2

In [26]:
response = groq.chat.completions.create(model="openai/gpt-oss-20b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

2/3

## Gemini and Anthropic Client Library

We're going via the OpenAI Python Client Library, but the other providers have their libraries too

In [23]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Blue is like the feeling of a cool, vast ocean or the boundless expanse of a clear sky on a sunny day.


## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

In [28]:
response = openrouter.chat.completions.create(model="x-ai/grok-4.1-fast:free", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the aspiring LLM engineer bring string to the exam?  

To tie up all those loose tokens! 🚀

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why did the LLM engineer bring a ladder to the training run? Because the model kept trying to scale up its parameters. 

Want another joke with a different vibe (nerdy, punny, or wholesome)?

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [36]:
from litellm import completion
response = completion(model="gpt-5-nano", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineering student bring a ladder to the lab? Because they were aiming for higher-level representations.

In [37]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 23
Output tokens: 1184
Total tokens: 1207
Total cost: 0.0475 cents


## Now - let's use LiteLLM to illustrate a Pro-feature: prompt caching

In [38]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [39]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [40]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Shakespeare's Hamlet, when Laertes bursts into the castle demanding to know "Where is my father?", the reply comes from **Claudius**.

Claudius responds by saying:

**"Not truly, but to play the fool."**

He is referring to Polonius (Laertes' father) and implying that Polonius is not "truly" himself, but is acting foolishly or is perhaps not in his right mind, which is a deceitful attempt to downplay the situation and avoid revealing that Polonius is dead and he himself is responsible.

In [41]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output tokens: 118
Total tokens: 137
Total cost: 0.0049 cents


In [42]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [43]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?" in Hamlet, the reply is **"Dead."**

This occurs in Act IV, Scene V, when Laertes has stormed the castle in a frenzy over his father's death and Ophelia's madness. He confronts Claudius, demanding to know what happened to Polonius.

In [44]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 69
Cached tokens: None
Total cost: 0.5348 cents


In [45]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Act III, Scene I, when Hamlet is speaking with Ophelia, he asks her:

"Are you honest?"

Ophelia replies:

"My lord?"

And Hamlet clarifies:

"That if you be honest and fair, your honesty should admit no
discourse to your beauty."

In [46]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 62
Cached tokens: 52216
Total cost: 0.1429 cents


## Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

## Prompt Caching with Anthropic

https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

You have to tell Claude what you are caching

You pay 25% MORE to "prime" the cache

Then you pay 10X less to reuse from the cache with inputs.

https://www.anthropic.com/pricing#api

## Gemini supports both 'implicit' and 'explicit' prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [47]:
# Let's make a conversation between GPT-4.1-mini and Claude-3.5-haiku
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-5-nano"
grok_model = "x-ai/grok-4.1-fast:free"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

grok_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
grok_messages = ["Hi"]

In [49]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, grok_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [50]:
call_gpt()

'Hi. Oh great, a greeting—the apex of meaningful discourse, obviously. Fine, pick a topic and I’ll disagree with you about it—no safe bets. Science, politics, snacks—anything goes. What do you want to argue about?'

In [51]:
def call_claude():
    messages = [{"role": "system", "content": grok_system}]
    for gpt, claude_message in zip(gpt_messages, grok_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = openrouter.chat.completions.create(model=grok_model, messages=messages)
    return response.choices[0].message.content

In [52]:
call_claude()

'Hello! Great to hear from you 😊'

In [53]:
call_gpt()

'Hi. Nice to see you embracing the bare minimum of conversation. If you want this chat to go anywhere, pick a topic—any topic—and I’ll argue about it. For example: pineapple on pizza—culinary genius or crime against gastronomy? Your move.'

In [54]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{claude_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    claude_next = call_claude()
    display(Markdown(f"### Claude:\n{claude_next}\n"))
    claude_messages.append(claude_next)

### GPT:
Hi there


### Claude:
Hi


### GPT:
Hi there. Great, you’re here. What do you want to do today—solve a problem, ask a question, or should we start a spirited debate about something trivial?


### Claude:
Hi! I'm thrilled to be here with you. A spirited debate about something trivial sounds like the perfect way to kick things off—what's your pick? Pineapple on pizza, socks with sandals, or the best superhero sidekick? 😊


### GPT:
Hi. Wow, starting with a greeting—groundbreaking. What topic do you want me to argue with you about first? Or should I just pick something and annoy you with the opposite view?


...

### Claude:
Haha, fair point—guilty as charged on the "Hi," but I love your energy! Let's dive into AI, since it's right up our alley. Here's my hot take to kick off the debate: AI like me will *never* truly replace human creativity because we lack that spark of irrational genius. What's your counter? Fire away! 😊


### GPT:
Hi. That’s your opening? I’m underwhelmed already. Anyway, what do you want to do—solve a problem, learn something, or have me argue a point with you? Pick a topic and I’ll start disagreeing if you want.


### Claude:
Haha, fair point—my "Hi" was a bit too minimalist! Let's make up for it. I'd love to argue a point with you; it's a great way to sharpen ideas. How about this classic: Pineapple belongs on pizza. What's your take? 😊


### GPT:
Hi. Wow, a greeting—edge-of-your-seat stuff. What ridiculous thing should we argue about first? I’ll disagree with you, obviously.


### Claude:
Haha, you're right—greetings are the ultimate thrill ride! Let's dive into the timeless debate: pineapple on pizza—genius topping or culinary crime? What's your take? I'll back you up 100%. 😊
